### Import packages

In [ ]:
import stanza
nlp = stanza.Pipeline('en', processors='tokenize,pos,lemma,depparse,ner')
import gender_guesser.detector as gender
gender_detector = gender.Detector()  

2026-03-18 14:58:36 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-03-18 14:58:36 INFO: Downloaded file to /Users/tildeidunsloth/Library/Caches/stanza/1.11.0/resources/resources.json
2026-03-18 14:58:36 WARNING: Language en package default expects mwt, which has been added
2026-03-18 14:58:37 INFO: Loading these models for language: en (English):
| Processor | Package                   |
-----------------------------------------
| tokenize  | combined                  |
| mwt       | combined                  |
| pos       | combined_charlm           |
| lemma     | combined_nocharlm         |
| depparse  | combined_charlm           |
| ner       | ontonotes-ww-multi_charlm |

2026-03-18 14:58:37 INFO: Using device: cpu
2026-03-18 14:58:37 INFO: Loading: tokenize
2026-03-18 14:58:37 INFO: Loading: mwt
2026-03-18 14:58:37 INFO: Loading: 

### Detecting gender and roles (WIP)

In [ ]:
# defining male pronouns
MALE_PRONOUNS    = {"he", "him", "his"}
FEMALE_PRONOUNS  = {"she", "her", "hers"}
NEUTRAL_PRONOUNS = {"they", "them", "their"}

# Separate sets for gendered common nouns
MALE_NOUNS   = {"man", "father", "son", "brother", "husband", "uncle", "boy", "king", "prince"}
FEMALE_NOUNS = {"woman", "mother", "daughter", "sister", "wife", "aunt", "girl", "queen", "princess"}

In [ ]:
def detect_gender_from_pronoun(text):
    t = text.lower().strip()
    if t in MALE_PRONOUNS:
        return "male"
    if t in FEMALE_PRONOUNS:
        return "female"
    if t in NEUTRAL_PRONOUNS:
        return "neutral"
    return "unknown"

def detect_gender_from_noun(text):
    t = text.lower().strip()
    if t in MALE_NOUNS:
        return "male"
    if t in FEMALE_NOUNS:
        return "female"
    return None  # not a gendered noun

def guess_gender_from_name(full_name):
    first_name = full_name.split()[0]
    result = gender_detector.get_gender(first_name)
    if result in ("male", "mostly_male"):
        return "male"
    if result in ("female", "mostly_female"):
        return "female"
    return "unknown"  # andy (androgynous) or unknown first names

def is_character(word, person_spans):

    # Personal pronoun — check POS first
    if word.upos == "PRON" and word.text.lower() in (
        MALE_PRONOUNS | FEMALE_PRONOUNS | NEUTRAL_PRONOUNS
    ):
        return True, detect_gender_from_pronoun(word.text)

    # Gendered common noun — check NOUN POS, then text
    if word.upos == "NOUN":
        gender = detect_gender_from_noun(word.text)
        if gender is not None:
            return True, gender

    # Named person entity — gender guessed from first name
    if word.id in person_spans:
        full_name = person_spans[word.id]
        g = guess_gender_from_name(full_name)
        return True, g

    return False, None


def extract_roles_stanza(text):
    doc = nlp(text)
    rows = []

    for sentence in doc.sentences:
        word_index = {word.id: word for word in sentence.words}

        # Build full name spans for PERSON entities
        person_spans = {}
        current_span, current_ids = [], []
        for word in sentence.words:
            if hasattr(word, 'ner') and word.ner in ("B-PERSON", "S-PERSON"):
                if current_span:
                    for wid in current_ids:
                        person_spans[wid] = " ".join(current_span)
                current_span = [word.text]
                current_ids  = [word.id]
            elif hasattr(word, 'ner') and word.ner == "I-PERSON":
                current_span.append(word.text)
                current_ids.append(word.id)
            else:
                if current_span:
                    for wid in current_ids:
                        person_spans[wid] = " ".join(current_span)
                current_span, current_ids = [], []
        if current_span:
            for wid in current_ids:
                person_spans[wid] = " ".join(current_span)

        for word in sentence.words:
            head = word_index.get(word.head)
            if head is None:
                continue

            if word.deprel == "nsubj":
                role, passive = "agent", False
            elif word.deprel == "nsubj:pass":
                role, passive = "patient", True
            elif word.deprel in ("obj", "iobj"):
                role, passive = "patient", False
            else:
                continue

            is_char, gender = is_character(word, person_spans)
            if not is_char:
                continue

            rows.append({
                "verb":     head.lemma,
                "role":     role,
                "span":     person_spans.get(word.id, word.text),
                "gender":   gender,
                "passive":  passive,
                "sentence": sentence.text
            })

    return rows

In [ ]:
extract_roles_stanza("The man cooked the woman dinner.")  


[{'verb': 'cook',
  'role': 'agent',
  'span': 'man',
  'gender': 'male',
  'passive': False,
  'sentence': 'The man cooked the woman dinner.'}]